In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
from datetime import date, datetime
import random
import os
import holidays

Mounted at /content/drive


In [ ]:
path = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset_baru/"
files_csv = os.listdir(path)

tables = {}

for file in files_csv:
    path_file = os.path.join(path, file)
    file_df = pd.read_csv(path_file)
    name_file = file.split(".")[1]

    tables[name_file] = file_df

tables.keys

<function dict.keys>

In [ ]:
tables.keys()

dict_keys(['orders', ' order_items', ' order_status_history', ' packing_qc', ' reject', ' returns', ' return_items', ' return_history', ' inventory_transaction', ' current_stock', ' purchase_orders', ' po_items', ' stores', ' products', ' catalog_items', ' catalog_product', ' employees', ' reject_history', ' suppliers'])

In [ ]:
orders_df = tables['orders']
order_items_df = tables[' order_items']
order_status_history_df = tables[' order_status_history']
packing_qc_df = tables[' packing_qc']
reject_df = tables[' reject']
returns_df = tables[' returns']
return_items_df = tables[' return_items']
return_history_df = tables[' return_history']
inventory_transaction_df = tables[' inventory_transaction']
current_stock_df = tables[' current_stock']
purchase_orders_df = tables[' purchase_orders']
po_items_df = tables[' po_items']
stores_df = tables[' stores']
products_df = tables[' products']
catalog_items_df = tables[' catalog_items']
catalog_product_df = tables[' catalog_product']
employees_df = tables[' employees']
suppliers_df = tables[' suppliers']
reject_history_df = tables[' reject_history']

# Master Data Orders

In [ ]:
# 1. Merge Awal (Tanpa perlu menarik kolom tax dari catalog, biar lebih ringan)
temp_mdo = pd.merge(
    orders_df[['order_id', 'date', 'status_order', 'platform', 'store_name', 'month']],
    order_items_df[['order_details_id', 'order_id', 'catalog_product_id', 'quantity', 'price']],
    how='left',
    on='order_id'
).rename(columns={'price': 'catalog_transaction_price'})

temp_mdo = pd.merge(
    temp_mdo,
    catalog_product_df[['catalog_product_id', 'product_name', 'cost_price']],
    how='left',
    on='catalog_product_id'
).rename(columns={
    'product_name': 'catalog_title',
    'cost_price': 'catalog_cost_price'
})

temp_mdo = pd.merge(
    temp_mdo,
    pd.merge(
        catalog_items_df[['catalog_product_id', 'product_id']],
        products_df[['product_id', 'product_name', 'category', 'vendor_id', 'cost_price']],
        how='left',
        on='product_id'
    ).rename(columns={'cost_price': 'product_cost_price'}),
    how='left',
    on='catalog_product_id'
)

temp_mdo = pd.merge(
    temp_mdo,
    suppliers_df[['supplier_id', 'supplier_name']],
    how='left',
    left_on='vendor_id',
    right_on='supplier_id'
)

# -------------------------------------------------------------------
# 2. PROSES ALOKASI HARGA & KALKULASI TAX
# -------------------------------------------------------------------
# Hitung bobot item terhadap total katalog
temp_mdo['weight_ratio'] = temp_mdo['product_cost_price'] / temp_mdo['catalog_cost_price'].replace(0, 1)

# Hitung Harga Jual Item secara proporsional
temp_mdo['allocated_selling_price'] = temp_mdo['weight_ratio'] * temp_mdo['catalog_transaction_price']

# KALKULASI TAX BARU (Sesuai fungsi: 12% dari selling price)
temp_mdo['product_tax'] = temp_mdo['allocated_selling_price'] * 0.12


# -------------------------------------------------------------------
# 3. AGREGASI FINAL
# -------------------------------------------------------------------
master_data_orders = temp_mdo.groupby(
    ['order_id', 'product_id', 'date', 'month', 'status_order', 'platform',
     'store_name', 'product_name', 'catalog_title', 'supplier_name', 'category']
).agg(
    total_quantity=('quantity', 'sum'),
    product_selling_price=('allocated_selling_price', 'max'),
    product_cost_price=('product_cost_price', 'max'),
    product_tax=('product_tax', 'max') # Pajak 12% dari allocated_selling_price
).reset_index()

# Tampilkan Hasil
master_data_orders

,order_id,product_id,date,month,status_order,platform,store_name,product_name,catalog_title,supplier_name,category,total_quantity,product_selling_price,product_cost_price,product_tax
0,ORD-1000-102-29285,PRD-8003-895-020,2022-01-11 12:48:00,2022-01,completed,shopee,Bogo Helmet,accessories flat clear,helm bogo grey gloss + flat clear,PT Optik Visor Jaya,accessories,1,33600.0,24000.0,4032.0
1,ORD-1000-102-29285,PRD-8874-102-001,2022-01-11 12:48:00,2022-01,completed,shopee,Bogo Helmet,helm bogo grey gloss,helm bogo grey gloss + flat clear,PT Bogo Nusantara Safety,helm bogo,1,270200.0,193000.0,32424.0
2,ORD-1000-111-122718,PRD-6900-776-019,2023-03-29 14:09:00,2023-03,completed,tiktok,Bogo Store,accessories cembung smoke,helm bogo sage + cembung smoke,PT Optik Visor Jaya,accessories,1,33600.0,24000.0,4032.0
3,ORD-1000-111-122718,PRD-7442-689-005,2023-03-29 14:09:00,2023-03,completed,tiktok,Bogo Store,helm bogo sage,helm bogo sage + cembung smoke,PT Bogo Nusantara Safety,helm bogo,1,270200.0,193000.0,32424.0
4,ORD-1000-111-122718,PRD-8774-995-029,2023-03-29 14:09:00,2023-03,completed,tiktok,Bogo Store,helm hijab bogo hitam doff,helm hijab bogo hitam doff,PT Syar'i Ride Apparel,helm hijab bogo,1,282800.0,202000.0,33936.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
588010,ORD-9999-937-202464,PRD-3442-197-008,2024-04-05 13:32:00,2024-04,completed,tiktok,Bogo Store,helm bogo kuning,helm bogo kuning + flat smoke,PT Bogo Nusantara Safety,helm bogo,1,270200.0,193000.0,32424.0
588011,ORD-9999-937-202464,PRD-8983-302-018,2024-04-05 13:32:00,2024-04,completed,tiktok,Bogo Store,accessories flat smoke,helm bogo kuning + flat smoke,PT Optik Visor Jaya,accessories,1,33600.0,24000.0,4032.0
588012,ORD-9999-951-304791,PRD-6543-518-038,2025-07-22 12:10:00,2025-07,completed,tiktok,Bogo Utama,helm hijab bogo pink soft,helm hijab bogo pink soft,PT Syar'i Ride Apparel,helm hijab bogo,1,282800.0,202000.0,33936.0
588013,ORD-9999-957-28935,PRD-8665-718-015,2022-01-09 14:10:00,2022-01,completed,tiktok,Bogo Utama,helm bogo darkgrey gloss,helm bogo darkgrey gloss + flat smoke,PT Bogo Nusantara Safety,helm bogo,1,270200.0,193000.0,32424.0


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/master_data_orders.csv"
master_data_orders.to_csv(path_file)

# Master Data Products QC

In [ ]:
# 1. Siapkan status riwayat terakhir dari reject_history_df
latest_history = reject_history_df.sort_values('timestamp').drop_duplicates('reject_id', keep='last')

# 2. Mulai penggabungan dengan packing_qc_df sebagai MASTER (paling kiri)
master_qc_df = pd.merge(
    packing_qc_df[['qc_id', 'order_id', 'product_id', 'attempt_no', 'status', 'date_qc']],

    # Join ke reject_df (hanya yang reject yang akan punya isi, yang pass akan NaN)
    reject_df[['qc_id', 'reject_id', 'reject_type', 'reason']],
    how='left',
    on='qc_id'
).merge(
    # Join ke history terakhir untuk tahu nasib akhir barang yang reject
    latest_history[['reject_id', 'status', 'timestamp']],
    how='left',
    on='reject_id'
).merge(
    # Join ke products_df untuk mendapatkan nama produk
    products_df[['product_id', 'product_name', 'category']],
    how='left',
    on='product_id'
)

# Ganti nama kolom agar lebih intuitif saat dibaca
master_qc_df.rename(columns={
    'status_y': 'final_reject_status',
    'status_x': 'qc_status',
    'timestamp': 'last_handled_date'
}, inplace=True)

# Tampilkan sampel data
master_qc_df

,qc_id,order_id,product_id,attempt_no,qc_status,date_qc,reject_id,reject_type,reason,final_reject_status,last_handled_date,product_name,category
0,QC-2535-323-0001,ORD-2679-792-0001,PRD-2684-647-003,1,reject,2021-08-03 16:29:00,RJT-4257-833-0001,qc,produk rusak,replacement_by_supplier,2021-08-04 17:00:00,helm bogo hitam gloss,helm bogo
1,QC-5557-928-0002,ORD-2679-792-0001,PRD-9283-272-023,1,pass,2021-08-03 16:29:00,NaN,NaN,NaN,NaN,NaN,accessories pet gloss,accessories
2,QC-3615-814-0003,ORD-2679-792-0001,PRD-2684-647-003,2,pass,2021-08-03 16:39:00,NaN,NaN,NaN,NaN,NaN,helm bogo hitam gloss,helm bogo
3,QC-5803-949-0004,ORD-5333-926-0002,PRD-8003-895-020,1,pass,2021-08-03 10:32:00,NaN,NaN,NaN,NaN,NaN,accessories flat clear,accessories
4,QC-6925-691-0005,ORD-5333-926-0002,PRD-3442-197-008,1,reject,2021-08-03 10:32:00,RJT-1750-777-0002,qc,produk rusak,replacement_by_supplier,2021-08-04 17:00:00,helm bogo kuning,helm bogo
...,...,...,...,...,...,...,...,...,...,...,...,...,...
783984,QC-7063-126-783985,ORD-6458-186-306993,PRD-8983-302-018,3,pass,2025-08-02 14:55:00,NaN,NaN,NaN,NaN,NaN,accessories flat smoke,accessories
783985,QC-7426-911-783986,ORD-5827-165-306994,PRD-9499-652-004,1,pass,2025-08-02 17:57:00,NaN,NaN,NaN,NaN,NaN,helm bogo mint,helm bogo
783986,QC-9632-900-783987,ORD-5827-165-306994,PRD-9283-272-023,1,reject,2025-08-02 17:57:00,RJT-9829-991-202153,qc,part produk tidak berfungsi,disposed,2025-08-02 17:57:00,accessories pet gloss,accessories
783987,QC-3451-920-783988,ORD-5827-165-306994,PRD-9283-272-023,2,pass,2025-08-02 18:07:00,NaN,NaN,NaN,NaN,NaN,accessories pet gloss,accessories


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/master_data_products_qc.csv"
master_qc_df.to_csv(path_file)

# Master Data Inventory Transaction

## MDIT Orders

In [ ]:
inv_order = inventory_transaction_df[inventory_transaction_df['reference']== 'order']

mdit_orders = pd.merge(
    inv_order,
    orders_df[['order_id', 'platform','store_name', 'month']],
    how='left',
    left_on='reference_id',
    right_on='order_id'
).merge(
    products_df[['product_id', 'product_name', 'category']],
    how='left',
    on='product_id'
).drop(columns='order_id')

mdit_orders

,transaction_id,product_id,date,quantity,type,actors_id,partners_id,explanation,created_at,created_by,reference,reference_id,platform,store_name,month,product_name,category
0,TRX-5552-259-0002,PRD-2684-647-003,2021-08-03 16:49:00,1,OUT,EMP-9316-263-002,STR-2673-270-001,qc_pass_ready_to_ship,2021-08-03 16:19:00,EMP-9316-263-002,order,ORD-2679-792-0001,tiktok,Bogo Store,2021-08,helm bogo hitam gloss,helm bogo
1,TRX-4527-881-0003,PRD-9283-272-023,2021-08-03 16:49:00,1,OUT,EMP-9316-263-002,STR-2673-270-001,qc_pass_ready_to_ship,2021-08-03 16:19:00,EMP-9316-263-002,order,ORD-2679-792-0001,tiktok,Bogo Store,2021-08,accessories pet gloss,accessories
2,TRX-2169-723-0006,PRD-3442-197-008,2021-08-03 11:02:00,1,OUT,EMP-9316-263-002,STR-2673-270-001,qc_pass_ready_to_ship,2021-08-03 10:22:00,EMP-9316-263-002,order,ORD-5333-926-0002,tiktok,Bogo Store,2021-08,helm bogo kuning,helm bogo
3,TRX-3803-646-0007,PRD-8003-895-020,2021-08-03 11:02:00,1,OUT,EMP-9316-263-002,STR-2673-270-001,qc_pass_ready_to_ship,2021-08-03 10:22:00,EMP-9316-263-002,order,ORD-5333-926-0002,tiktok,Bogo Store,2021-08,accessories flat clear,accessories
4,TRX-1188-796-0011,PRD-6689-748-014,2021-08-03 13:26:00,1,OUT,EMP-9316-263-002,STR-4301-394-002,qc_pass_ready_to_ship,2021-08-03 12:46:00,EMP-9316-263-002,order,ORD-4598-801-0003,shopee,Bogo Helmet,2021-08,helm bogo silver,helm bogo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
588817,TRX-3723-220-914737,PRD-8874-102-001,2025-08-02 15:05:00,1,OUT,EMP-9316-263-002,STR-2673-270-001,qc_pass_ready_to_ship,2025-08-02 14:25:00,EMP-9316-263-002,order,ORD-6458-186-306993,tiktok,Bogo Store,2025-08,helm bogo grey gloss,helm bogo
588818,TRX-8654-424-914738,PRD-6900-776-019,2025-08-02 15:05:00,1,OUT,EMP-9316-263-002,STR-2673-270-001,qc_pass_ready_to_ship,2025-08-02 14:25:00,EMP-9316-263-002,order,ORD-6458-186-306993,tiktok,Bogo Store,2025-08,accessories cembung smoke,accessories
588819,TRX-2956-423-914740,PRD-9499-652-004,2025-08-02 18:17:00,1,OUT,EMP-9316-263-002,STR-4301-394-002,qc_pass_ready_to_ship,2025-08-02 17:47:00,EMP-9316-263-002,order,ORD-5827-165-306994,shopee,Bogo Helmet,2025-08,helm bogo mint,helm bogo
588820,TRX-9278-264-914741,PRD-9283-272-023,2025-08-02 18:17:00,1,OUT,EMP-9316-263-002,STR-4301-394-002,qc_pass_ready_to_ship,2025-08-02 17:47:00,EMP-9316-263-002,order,ORD-5827-165-306994,shopee,Bogo Helmet,2025-08,accessories pet gloss,accessories


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/mdit_orders.csv"
mdit_orders.to_csv(path_file)

## MDIT Reject

In [ ]:
inv_reject = inventory_transaction_df[inventory_transaction_df['reference'] == 'reject']

mdit_rejects = pd.merge(
    inv_reject,
    reject_df[['reject_id', 'qc_id', 'order_id', 'status', 'reason']], # reason milik tabel reject
    left_on='reference_id',
    right_on='reject_id',
    how='left'
).merge(
    packing_qc_df[['qc_id', 'attempt_no']], # Sesuaikan nama kolom aslinya
    on='qc_id',
    how='left'
).merge(
    products_df[['product_id', 'product_name']], # Tambahan kecil biar langsung tau nama barangnya
    on='product_id',
    how='left'
).drop(columns=['reject_id']) # Bersihkan kolom duplikat dari hasil left_on & right_on
mdit_rejects

,transaction_id,product_id,date,quantity,type,actors_id,partners_id,explanation,created_at,created_by,reference,reference_id,qc_id,order_id,status,reason,attempt_no,product_name
0,TRX-9928-529-0001,PRD-2684-647-003,2021-08-03 16:29:00,1,OUT,EMP-9316-263-002,SPLR-6274-501-001,qc_reject,2021-08-03 16:29:00,EMP-9316-263-002,reject,RJT-4257-833-0001,QC-2535-323-0001,ORD-2679-792-0001,replacement_by_supplier,produk rusak,1,helm bogo hitam gloss
1,TRX-4733-891-0004,PRD-3442-197-008,2021-08-03 10:32:00,1,OUT,EMP-9316-263-002,SPLR-6274-501-001,qc_reject,2021-08-03 10:32:00,EMP-9316-263-002,reject,RJT-1750-777-0002,QC-6925-691-0005,ORD-5333-926-0002,replacement_by_supplier,produk rusak,1,helm bogo kuning
2,TRX-6977-266-0005,PRD-3442-197-008,2021-08-03 10:42:00,1,OUT,EMP-9316-263-002,SPLR-6274-501-001,qc_reject,2021-08-03 10:42:00,EMP-9316-263-002,reject,RJT-8428-750-0003,QC-4814-987-0006,ORD-5333-926-0002,replacement_by_supplier,Komponen produk rusak,2,helm bogo kuning
3,TRX-9179-505-0008,PRD-6689-748-014,2021-08-03 12:56:00,1,OUT,EMP-9316-263-002,SPLR-6274-501-001,qc_reject,2021-08-03 12:56:00,EMP-9316-263-002,reject,RJT-4483-771-0004,QC-7572-374-0008,ORD-4598-801-0003,replacement_by_supplier,Komponen produk rusak,1,helm bogo silver
4,TRX-7543-470-0009,PRD-8164-914-021,2021-08-03 12:56:00,1,OUT,EMP-9316-263-002,SPLR-6208-341-002,qc_reject,2021-08-03 12:56:00,EMP-9316-263-002,reject,RJT-8019-697-0005,QC-5339-242-0009,ORD-4598-801-0003,disposed,Komponen produk rusak,1,accessories flat rainbow
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
310705,TRX-2266-267-914810,PRD-8338-913-043,2025-08-02 17:00:00,1,IN,EMP-4124-636-003,SPLR-6297-996-004,replacement_from_supplier,2025-08-02 17:00:00,EMP-1998-144-001,reject,RJT-2861-302-201976,QC-7405-704-783358,ORD-3428-261-306752,replacement_by_supplier,ada noda yg tidak bisa hilang,1,helm cakil putih gloss
310706,TRX-7106-146-914811,PRD-9823-413-041,2025-08-02 17:00:00,1,IN,EMP-4124-636-003,SPLR-6297-996-004,replacement_from_supplier,2025-08-02 17:00:00,EMP-1998-144-001,reject,RJT-9603-130-201912,QC-9274-791-783127,ORD-3346-113-306666,replacement_by_supplier,ada noda yg tidak bisa hilang,1,helm cakil hitam dof
310707,TRX-3676-703-914812,PRD-9823-413-041,2025-08-02 17:00:00,1,IN,EMP-4124-636-003,SPLR-6297-996-004,replacement_from_supplier,2025-08-02 17:00:00,EMP-1998-144-001,reject,RJT-2709-670-201958,QC-8463-766-783289,ORD-2192-332-306726,replacement_by_supplier,Komponen produk rusak,1,helm cakil hitam dof
310708,TRX-5202-375-914813,PRD-2320-155-044,2025-08-02 17:00:00,1,IN,EMP-4124-636-003,SPLR-6297-996-004,replacement_from_supplier,2025-08-02 17:00:00,EMP-1998-144-001,reject,RJT-6134-997-201926,QC-7309-928-783180,ORD-1045-508-306686,replacement_by_supplier,ada noda yg tidak bisa hilang,1,helm cakil darkgrey gloss


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/mdit_rejects.csv"
mdit_rejects.to_csv(path_file)

## MDIT Purchase

In [ ]:
# 1. Filter khusus transaksi barang masuk dari supplier
inv_purchase = inventory_transaction_df[inventory_transaction_df['reference'] == 'purchase']

# 2. Join ke Header PO untuk dapat tanggal dan status PO
mdit_purchase = pd.merge(
    inv_purchase,
    purchase_orders_df[['po_id', 'order_date', 'status']], # Sesuaikan nama kolom aslinya
    left_on='reference_id',
    right_on='po_id',
    how='left'
).merge(
    # 3. Join ke Detail PO untuk dapat harga modal dan qty pesanan
    # Gunakan 2 key agar tidak salah mencocokkan barang
    po_items_df[['po_id', 'product_id', 'quantity', 'cost_price']], # Asumsi ada kolom price/cost di sini
    left_on=['po_id', 'product_id'],
    right_on=['po_id', 'product_id'],
    how='left',
    suffixes=('_received', '_ordered') # SUPER PENTING: Mencegah nama kolom 'quantity' bentrok
).drop(columns=['po_id']) # Bersihkan kolom duplikat

mdit_purchase

,transaction_id,product_id,date,quantity_received,type,actors_id,partners_id,explanation,created_at,created_by,reference,reference_id,order_date,status,quantity_ordered,cost_price
0,TRX-8571-882-001,PRD-8874-102-001,2021-08-03 07:10:30,50,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001,2021-08-03 07:10:30,ordered,50,193000.0
1,TRX-5224-208-002,PRD-5383-617-002,2021-08-03 07:10:30,100,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001,2021-08-03 07:10:30,ordered,100,193000.0
2,TRX-5629-903-003,PRD-2684-647-003,2021-08-03 07:10:30,50,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001,2021-08-03 07:10:30,ordered,50,193000.0
3,TRX-2322-258-004,PRD-9499-652-004,2021-08-03 07:10:30,20,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001,2021-08-03 07:10:30,ordered,20,193000.0
4,TRX-9209-447-005,PRD-7442-689-005,2021-08-03 07:10:30,20,IN,EMP-4124-636-003,SPLR-6274-501-001,Initial_stock,2021-08-03 07:10:30,EMP-1998-144-001,purchase,PO-2826-234-001,2021-08-03 07:10:30,ordered,20,193000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5804,TRX-8040-531-914202,PRD-9825-531-012,2025-08-01 17:00:00,20,IN,EMP-4124-636-003,SPLR-6274-501-001,restock,2025-08-01 17:00:00,EMP-1998-144-001,purchase,PO-4236-657-4669,2025-08-01 17:00:00,received,20,193000.0
5805,TRX-9748-655-914792,PRD-7442-689-005,2025-08-02 17:00:00,20,IN,EMP-4124-636-003,SPLR-6274-501-001,restock,2025-08-02 17:00:00,EMP-1998-144-001,purchase,PO-2671-289-4672,2025-08-02 17:00:00,received,20,193000.0
5806,TRX-7261-554-914796,PRD-9499-652-004,2025-08-02 17:00:00,20,IN,EMP-4124-636-003,SPLR-6274-501-001,restock,2025-08-02 17:00:00,EMP-1998-144-001,purchase,PO-2671-289-4672,2025-08-02 17:00:00,received,20,193000.0
5807,TRX-8781-965-914800,PRD-1921-457-011,2025-08-02 17:00:00,50,IN,EMP-4124-636-003,SPLR-6274-501-001,restock,2025-08-02 17:00:00,EMP-1998-144-001,purchase,PO-2671-289-4672,2025-08-02 17:00:00,received,50,193000.0


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/mdit_purchase.csv"
mdit_purchase.to_csv(path_file)

## MDIT Return

In [ ]:
inv_return = inventory_transaction_df[inventory_transaction_df['reference'] == 'return']

inv_return

,transaction_id,product_id,date,quantity,type,actors_id,partners_id,explanation,created_at,created_by,reference,reference_id
3094,TRX-5803-392-3050,PRD-9283-272-023,2021-08-09 13:00:00,1,IN,EMP-4124-636-003,STR-2673-270-001,return,2021-08-09 13:00:00,EMP-1998-144-001,return,RETI-3307-593-0002
3095,TRX-4157-738-3051,PRD-3442-197-008,2021-08-09 13:00:00,1,IN,EMP-4124-636-003,STR-2673-270-001,return,2021-08-09 13:00:00,EMP-1998-144-001,return,RETI-3017-470-0003
3096,TRX-1085-960-3052,PRD-8338-913-043,2021-08-09 13:00:00,1,IN,EMP-4124-636-003,STR-4301-394-002,return,2021-08-09 13:00:00,EMP-1998-144-001,return,RETI-9107-384-0005
3097,TRX-4055-245-3053,PRD-8983-302-018,2021-08-09 13:00:00,1,IN,EMP-4124-636-003,STR-4301-394-002,return,2021-08-09 13:00:00,EMP-1998-144-001,return,RETI-1988-270-0006
3698,TRX-8836-134-3654,PRD-9661-508-033,2021-08-10 13:00:00,1,IN,EMP-4124-636-003,STR-2673-270-001,return,2021-08-10 13:00:00,EMP-1998-144-001,return,RETI-3068-321-0009
...,...,...,...,...,...,...,...,...,...,...,...,...
913654,TRX-8856-636-913610,PRD-1921-457-011,2025-08-01 13:00:00,1,IN,EMP-4124-636-003,STR-4301-394-002,return,2025-08-01 13:00:00,EMP-1998-144-001,return,RETI-3449-991-11872
913655,TRX-9418-383-913611,PRD-8003-895-020,2025-08-01 13:00:00,1,IN,EMP-4124-636-003,STR-4301-394-002,return,2025-08-01 13:00:00,EMP-1998-144-001,return,RETI-3832-954-11873
913656,TRX-3755-582-913612,PRD-3442-197-008,2025-08-01 13:00:00,1,IN,EMP-4124-636-003,STR-8526-652-004,return,2025-08-01 13:00:00,EMP-1998-144-001,return,RETI-1978-746-11876
913657,TRX-5036-272-913613,PRD-8003-895-020,2025-08-01 13:00:00,1,IN,EMP-4124-636-003,STR-8526-652-004,return,2025-08-01 13:00:00,EMP-1998-144-001,return,RETI-3024-758-11877


In [ ]:
mdit_return = pd.merge(
    inv_return,
    return_items_df[['return_item_id','return_id', 'product_id', 'qty', 'condition', 'restockable']],
    how='left',
    left_on=['reference_id', 'product_id'],
    right_on=['return_item_id', 'product_id']
).merge(
    returns_df[['return_id','return_date', 'reason', 'refund_amount', 'status']],
    how='left',
    on='return_id'
)
mdit_return

,transaction_id,product_id,date,quantity,type,actors_id,partners_id,explanation,created_at,created_by,...,reference_id,return_item_id,return_id,qty,condition,restockable,return_date,reason,refund_amount,status
0,TRX-5803-392-3050,PRD-9283-272-023,2021-08-09 13:00:00,1,IN,EMP-4124-636-003,STR-2673-270-001,return,2021-08-09 13:00:00,EMP-1998-144-001,...,RETI-3307-593-0002,RETI-3307-593-0002,RET-7767-865-0001,1,good,yes,2021-08-06 14:00:00,warna tidak sesuai,303800.0,received
1,TRX-4157-738-3051,PRD-3442-197-008,2021-08-09 13:00:00,1,IN,EMP-4124-636-003,STR-2673-270-001,return,2021-08-09 13:00:00,EMP-1998-144-001,...,RETI-3017-470-0003,RETI-3017-470-0003,RET-5094-932-0002,1,good,yes,2021-08-06 14:00:00,produk tidak sesuai ukuran,303800.0,received
2,TRX-1085-960-3052,PRD-8338-913-043,2021-08-09 13:00:00,1,IN,EMP-4124-636-003,STR-4301-394-002,return,2021-08-09 13:00:00,EMP-1998-144-001,...,RETI-9107-384-0005,RETI-9107-384-0005,RET-7268-493-0003,1,good,yes,2021-08-06 14:00:00,produk rusak saat ekspedisi,301000.0,received
3,TRX-4055-245-3053,PRD-8983-302-018,2021-08-09 13:00:00,1,IN,EMP-4124-636-003,STR-4301-394-002,return,2021-08-09 13:00:00,EMP-1998-144-001,...,RETI-1988-270-0006,RETI-1988-270-0006,RET-7268-493-0003,1,good,yes,2021-08-06 14:00:00,produk rusak saat ekspedisi,301000.0,received
4,TRX-8836-134-3654,PRD-9661-508-033,2021-08-10 13:00:00,1,IN,EMP-4124-636-003,STR-2673-270-001,return,2021-08-10 13:00:00,EMP-1998-144-001,...,RETI-3068-321-0009,RETI-3068-321-0009,RET-5949-341-0005,1,good,yes,2021-08-07 14:00:00,produk rusak saat ekspedisi,282800.0,received
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9514,TRX-8856-636-913610,PRD-1921-457-011,2025-08-01 13:00:00,1,IN,EMP-4124-636-003,STR-4301-394-002,return,2025-08-01 13:00:00,EMP-1998-144-001,...,RETI-3449-991-11872,RETI-3449-991-11872,RET-5687-611-6129,1,good,yes,2025-07-29 14:00:00,produk rusak saat ekspedisi,303800.0,received
9515,TRX-9418-383-913611,PRD-8003-895-020,2025-08-01 13:00:00,1,IN,EMP-4124-636-003,STR-4301-394-002,return,2025-08-01 13:00:00,EMP-1998-144-001,...,RETI-3832-954-11873,RETI-3832-954-11873,RET-5687-611-6129,1,good,yes,2025-07-29 14:00:00,produk rusak saat ekspedisi,303800.0,received
9516,TRX-3755-582-913612,PRD-3442-197-008,2025-08-01 13:00:00,1,IN,EMP-4124-636-003,STR-8526-652-004,return,2025-08-01 13:00:00,EMP-1998-144-001,...,RETI-1978-746-11876,RETI-1978-746-11876,RET-6091-714-6130,1,good,yes,2025-07-29 14:00:00,produk tidak sesuai ukuran,607600.0,received
9517,TRX-5036-272-913613,PRD-8003-895-020,2025-08-01 13:00:00,1,IN,EMP-4124-636-003,STR-8526-652-004,return,2025-08-01 13:00:00,EMP-1998-144-001,...,RETI-3024-758-11877,RETI-3024-758-11877,RET-6091-714-6130,1,good,yes,2025-07-29 14:00:00,produk tidak sesuai ukuran,607600.0,received


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/mdit_return.csv"
mdit_return.to_csv(path_file)

# Master Data Orders-QC Packing

In [ ]:
master_data_orders.columns

Index(['order_id', 'product_id', 'date', 'month', 'status_order', 'platform',
       'store_name', 'product_name', 'catalog_title', 'supplier_name',
       'category', 'total_quantity', 'product_selling_price',
       'product_cost_price', 'product_tax'],
      dtype='object')

In [ ]:
# -------------------------------------------------------------------
# 1. PADATKAN MDO DULU (Hancurkan duplikasi karena catalog_title)
# -------------------------------------------------------------------
mdo_unique = master_data_orders.groupby([
    'order_id', 'product_id', 'date', 'month', 'product_name',
    'category', 'status_order', 'supplier_name', 'store_name'
]).agg(
    catalog_title=('catalog_title', lambda x: ', '.join(x.unique())), # Gabung kalau ada 2 katalog
    total_quantity=('total_quantity', 'sum'),
    product_selling_price=('product_selling_price', 'mean'),
    product_cost_price=('product_cost_price', 'first'),
    product_tax=('product_tax', 'mean')
).reset_index()


# -------------------------------------------------------------------
# 2. RANGKUM QC & REJECT (Sesuai kodemu, tapi perbaiki sum quantity-nya)
# -------------------------------------------------------------------
qc_summary = packing_qc_df.groupby(['order_id', 'product_id']).agg(
    total_qc_attempts=('attempt_no', 'max'),
    final_qc_status=('status', 'last'),
    qc_employee_id=('employee_id', 'last'),
).reset_index()

reject_summary = reject_df.groupby(['order_id', 'product_id']).agg(
    total_rejected_qty=('quantity', 'sum'),                     # <-- PERBAIKAN: Gunakan sum pada quantity!
    reject_reasons=('reason', lambda x: ', '.join(x.unique())),
    reject_status=('status', lambda x: ', '.join(x.unique()))
).reset_index()


# -------------------------------------------------------------------
# 3. MERGE SEMUANYA MENJADI MD_FULFILLMENT
# -------------------------------------------------------------------
md_fulfillment = pd.merge(mdo_unique, qc_summary, how='left', on=['order_id', 'product_id'])
md_fulfillment = pd.merge(md_fulfillment, reject_summary, how='left', on=['order_id', 'product_id'])

# 4. Cleaning nilai kosong
md_fulfillment['total_rejected_qty'] = md_fulfillment['total_rejected_qty'].fillna(0).astype(int)
md_fulfillment['reject_reasons'] = md_fulfillment['reject_reasons'].fillna('-')
md_fulfillment['reject_status'] = md_fulfillment['reject_status'].fillna('-')

# VERIFIKASI AKHIR
print("Asli dari reject_df     :", reject_df['quantity'].sum())
print("Total di md_fulfillment :", md_fulfillment['total_rejected_qty'].sum())

Asli dari reject_df     : 202153
Total di md_fulfillment : 202153


In [ ]:
md_fulfillment

,order_id,product_id,date,month,product_name,category,status_order,supplier_name,store_name,catalog_title,total_quantity,product_selling_price,product_cost_price,product_tax,total_qc_attempts,final_qc_status,qc_employee_id,total_rejected_qty,reject_reasons,reject_status
0,ORD-1000-102-29285,PRD-8003-895-020,2022-01-11 12:48:00,2022-01,accessories flat clear,accessories,completed,PT Optik Visor Jaya,Bogo Helmet,helm bogo grey gloss + flat clear,1,33600.0,24000.0,4032.0,1,pass,EMP-9316-263-002,0,-,-
1,ORD-1000-102-29285,PRD-8874-102-001,2022-01-11 12:48:00,2022-01,helm bogo grey gloss,helm bogo,completed,PT Bogo Nusantara Safety,Bogo Helmet,helm bogo grey gloss + flat clear,1,270200.0,193000.0,32424.0,1,pass,EMP-9316-263-002,0,-,-
2,ORD-1000-111-122718,PRD-6900-776-019,2023-03-29 14:09:00,2023-03,accessories cembung smoke,accessories,completed,PT Optik Visor Jaya,Bogo Store,helm bogo sage + cembung smoke,1,33600.0,24000.0,4032.0,1,pass,EMP-9316-263-002,0,-,-
3,ORD-1000-111-122718,PRD-7442-689-005,2023-03-29 14:09:00,2023-03,helm bogo sage,helm bogo,completed,PT Bogo Nusantara Safety,Bogo Store,helm bogo sage + cembung smoke,1,270200.0,193000.0,32424.0,1,pass,EMP-9316-263-002,0,-,-
4,ORD-1000-111-122718,PRD-8774-995-029,2023-03-29 14:09:00,2023-03,helm hijab bogo hitam doff,helm hijab bogo,completed,PT Syar'i Ride Apparel,Bogo Store,helm hijab bogo hitam doff,1,282800.0,202000.0,33936.0,1,pass,EMP-9316-263-002,0,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581831,ORD-9999-937-202464,PRD-3442-197-008,2024-04-05 13:32:00,2024-04,helm bogo kuning,helm bogo,completed,PT Bogo Nusantara Safety,Bogo Store,helm bogo kuning + flat smoke,1,270200.0,193000.0,32424.0,1,pass,EMP-9316-263-002,0,-,-
581832,ORD-9999-937-202464,PRD-8983-302-018,2024-04-05 13:32:00,2024-04,accessories flat smoke,accessories,completed,PT Optik Visor Jaya,Bogo Store,helm bogo kuning + flat smoke,1,33600.0,24000.0,4032.0,1,pass,EMP-9316-263-002,0,-,-
581833,ORD-9999-951-304791,PRD-6543-518-038,2025-07-22 12:10:00,2025-07,helm hijab bogo pink soft,helm hijab bogo,completed,PT Syar'i Ride Apparel,Bogo Utama,helm hijab bogo pink soft,1,282800.0,202000.0,33936.0,1,pass,EMP-9316-263-002,0,-,-
581834,ORD-9999-957-28935,PRD-8665-718-015,2022-01-09 14:10:00,2022-01,helm bogo darkgrey gloss,helm bogo,completed,PT Bogo Nusantara Safety,Bogo Utama,helm bogo darkgrey gloss + flat smoke,1,270200.0,193000.0,32424.0,2,pass,EMP-9316-263-002,1,part produk tidak berfungsi,replacement_by_supplier


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/md_orders_x_qc.csv"
md_fulfillment.to_csv(path_file)

# Master Data After Sales (Order x Return)

In [ ]:
# 1. Siapkan return_detail (Kondisi sudah benar)
return_detail = pd.merge(
    return_items_df,
    returns_df[['return_id', 'order_id', 'return_date', 'reason', 'refund_amount', 'status']],
    on='return_id',
    how='left'
)

return_detail = return_detail.rename(columns={
    'status': 'return_status',
    'reason': 'return_reason',
    'qty': 'qty_returned'
})


# Kita leburkan baris produk yang sama dalam 1 order
mdo_safe = master_data_orders.groupby(
    ['order_id', 'product_id', 'date', 'month', 'product_name', 'category', 'supplier_name', 'store_name']
).agg(               # AMAN: Kuantitas dijumlahkan
    product_selling_price=('product_selling_price', 'mean'),  # AMAN: Harga dirata-rata
    product_cost_price=('product_cost_price', 'first'),       # Modal pasti sama
    product_tax=('product_tax', 'mean')                       # Pajak dirata-rata
).reset_index()


# -------------------------------------------------------------------
# MERGE DENGAN DATA RETUR
# -------------------------------------------------------------------
md_after_sales = pd.merge(
    mdo_safe,
    return_detail[['order_id', 'product_id', 'return_id', 'return_date',
                   'qty_returned', 'condition', 'restockable', 'return_reason', 'return_status']],
    how='left',
    on=['order_id', 'product_id']
)

# Buat kolom flag/penanda
md_after_sales['is_returned'] = md_after_sales['return_id'].notna().astype(int)
md_after_sales['return_reason'] = md_after_sales['return_reason'].fillna('No Return')

# Verifikasi
print("Jumlah row is_returned == 1 :", md_after_sales['is_returned'].sum())

md_after_sales

Jumlah row is_returned == 1 : 11899


,order_id,product_id,date,month,product_name,category,supplier_name,store_name,product_selling_price,product_cost_price,product_tax,return_id,return_date,qty_returned,condition,restockable,return_reason,return_status,is_returned
0,ORD-1000-102-29285,PRD-8003-895-020,2022-01-11 12:48:00,2022-01,accessories flat clear,accessories,PT Optik Visor Jaya,Bogo Helmet,33600.0,24000.0,4032.0,NaN,NaN,NaN,NaN,NaN,No Return,NaN,0
1,ORD-1000-102-29285,PRD-8874-102-001,2022-01-11 12:48:00,2022-01,helm bogo grey gloss,helm bogo,PT Bogo Nusantara Safety,Bogo Helmet,270200.0,193000.0,32424.0,NaN,NaN,NaN,NaN,NaN,No Return,NaN,0
2,ORD-1000-111-122718,PRD-6900-776-019,2023-03-29 14:09:00,2023-03,accessories cembung smoke,accessories,PT Optik Visor Jaya,Bogo Store,33600.0,24000.0,4032.0,NaN,NaN,NaN,NaN,NaN,No Return,NaN,0
3,ORD-1000-111-122718,PRD-7442-689-005,2023-03-29 14:09:00,2023-03,helm bogo sage,helm bogo,PT Bogo Nusantara Safety,Bogo Store,270200.0,193000.0,32424.0,NaN,NaN,NaN,NaN,NaN,No Return,NaN,0
4,ORD-1000-111-122718,PRD-8774-995-029,2023-03-29 14:09:00,2023-03,helm hijab bogo hitam doff,helm hijab bogo,PT Syar'i Ride Apparel,Bogo Store,282800.0,202000.0,33936.0,NaN,NaN,NaN,NaN,NaN,No Return,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581995,ORD-9999-937-202464,PRD-3442-197-008,2024-04-05 13:32:00,2024-04,helm bogo kuning,helm bogo,PT Bogo Nusantara Safety,Bogo Store,270200.0,193000.0,32424.0,NaN,NaN,NaN,NaN,NaN,No Return,NaN,0
581996,ORD-9999-937-202464,PRD-8983-302-018,2024-04-05 13:32:00,2024-04,accessories flat smoke,accessories,PT Optik Visor Jaya,Bogo Store,33600.0,24000.0,4032.0,NaN,NaN,NaN,NaN,NaN,No Return,NaN,0
581997,ORD-9999-951-304791,PRD-6543-518-038,2025-07-22 12:10:00,2025-07,helm hijab bogo pink soft,helm hijab bogo,PT Syar'i Ride Apparel,Bogo Utama,282800.0,202000.0,33936.0,NaN,NaN,NaN,NaN,NaN,No Return,NaN,0
581998,ORD-9999-957-28935,PRD-8665-718-015,2022-01-09 14:10:00,2022-01,helm bogo darkgrey gloss,helm bogo,PT Bogo Nusantara Safety,Bogo Utama,270200.0,193000.0,32424.0,NaN,NaN,NaN,NaN,NaN,No Return,NaN,0


In [ ]:
md_after_sales[md_after_sales['is_returned'] == 1]

,order_id,product_id,date,month,product_name,category,supplier_name,store_name,product_selling_price,product_cost_price,product_tax,return_id,return_date,qty_returned,condition,restockable,return_reason,return_status,is_returned
238,ORD-1003-916-89645,PRD-8003-895-020,2022-10-25 11:18:00,2022-10,accessories flat clear,accessories,PT Optik Visor Jaya,Bogo Store,33600.0,24000.0,4032.0,RET-9351-958-1800,2022-10-28 14:00:00,1.0,good,yes,produk tidak sesuai ukuran,received,1
239,ORD-1003-916-89645,PRD-9290-842-017,2022-10-25 11:18:00,2022-10,helm bogo capuchino,helm bogo,PT Bogo Nusantara Safety,Bogo Store,270200.0,193000.0,32424.0,RET-9351-958-1800,2022-10-28 14:00:00,1.0,good,yes,produk tidak sesuai ukuran,received,1
288,ORD-1004-878-292873,PRD-7478-588-006,2025-05-26 13:37:00,2025-05,helm bogo hitam doff,helm bogo,PT Bogo Nusantara Safety,Bogo Store,270200.0,193000.0,32424.0,RET-5011-961-5916,2025-05-29 14:00:00,1.0,good,yes,warna tidak sesuai,received,1
289,ORD-1004-878-292873,PRD-8164-914-021,2025-05-26 13:37:00,2025-05,accessories flat rainbow,accessories,PT Optik Visor Jaya,Bogo Store,33600.0,24000.0,4032.0,RET-5011-961-5916,2025-05-29 14:00:00,1.0,good,yes,warna tidak sesuai,received,1
338,ORD-1005-643-245792,PRD-8003-895-020,2024-10-22 11:19:00,2024-10,accessories flat clear,accessories,PT Optik Visor Jaya,Bogo United,33600.0,24000.0,4032.0,RET-1304-326-5013,2024-10-25 14:00:00,1.0,good,yes,warna tidak sesuai,received,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581872,ORD-9998-142-34105,PRD-8266-654-028,2022-02-03 11:03:00,2022-02,helm hijab bogo sage,helm hijab bogo,PT Syar'i Ride Apparel,Bogo Helmet,282800.0,202000.0,33936.0,RET-9976-811-0685,2022-02-06 14:00:00,1.0,damaged,no,produk tidak sesuai ukuran,received,1
581873,ORD-9998-142-34105,PRD-9831-122-039,2022-02-03 11:03:00,2022-02,helm hijab bogo capuchino,helm hijab bogo,PT Syar'i Ride Apparel,Bogo Helmet,282800.0,202000.0,33936.0,RET-9976-811-0685,2022-02-06 14:00:00,1.0,good,yes,produk tidak sesuai ukuran,received,1
581933,ORD-9998-961-206898,PRD-5030-206-022,2024-04-20 09:58:00,2024-04,accessories pet doff,accessories,PT Optik Visor Jaya,Bogo Store,33600.0,24000.0,4032.0,RET-8235-199-4230,2024-04-23 14:00:00,1.0,good,yes,produk rusak saat ekspedisi,received,1
581934,ORD-9998-961-206898,PRD-5030-206-022,2024-04-20 09:58:00,2024-04,accessories pet doff,accessories,PT Optik Visor Jaya,Bogo Store,33600.0,24000.0,4032.0,RET-8235-199-4230,2024-04-23 14:00:00,1.0,good,yes,produk rusak saat ekspedisi,received,1


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/md_after_sales.csv"
md_after_sales.to_csv(path_file)

# Dim Star Schema

## Dim Products

In [ ]:
dim_products_df = pd.merge(
    products_df[['product_id', 'product_name', 'category', 'vendor_id', 'cost_price']],
    suppliers_df[['supplier_id', 'supplier_name']],
    how='left',
    left_on='vendor_id',
    right_on='supplier_id'
).drop(columns=['vendor_id', 'supplier_id'])

dim_products_df

,product_id,product_name,category,cost_price,supplier_name
0,PRD-8874-102-001,helm bogo grey gloss,helm bogo,193000.0,PT Bogo Nusantara Safety
1,PRD-5383-617-002,helm bogo grey doff,helm bogo,193000.0,PT Bogo Nusantara Safety
2,PRD-2684-647-003,helm bogo hitam gloss,helm bogo,193000.0,PT Bogo Nusantara Safety
3,PRD-9499-652-004,helm bogo mint,helm bogo,193000.0,PT Bogo Nusantara Safety
4,PRD-7442-689-005,helm bogo sage,helm bogo,193000.0,PT Bogo Nusantara Safety
5,PRD-7478-588-006,helm bogo hitam doff,helm bogo,193000.0,PT Bogo Nusantara Safety
6,PRD-4480-984-007,helm bogo bluesky,helm bogo,193000.0,PT Bogo Nusantara Safety
7,PRD-3442-197-008,helm bogo kuning,helm bogo,193000.0,PT Bogo Nusantara Safety
8,PRD-6679-849-009,helm bogo cream,helm bogo,193000.0,PT Bogo Nusantara Safety
9,PRD-2992-160-010,helm bogo army doff,helm bogo,193000.0,PT Bogo Nusantara Safety


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/dim_products.csv"
dim_products_df.to_csv(path_file)

## Dim Store

In [ ]:
dim_store_df = stores_df[['store_id', 'store_name', 'platform']]
dim_store_df

,store_id,store_name,platform
0,STR-2673-270-001,Bogo Store,tiktok
1,STR-4301-394-002,Bogo Helmet,shopee
2,STR-2839-533-003,Bogo Utama,tiktok
3,STR-8526-652-004,Bogo United,shopee


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/dim_stores.csv"
dim_store_df.to_csv(path_file)

## Dim Employees

In [ ]:
dim_employees_df = employees_df[['employee_id', 'employee_name', 'role']]
dim_employees_df

,employee_id,employee_name,role
0,EMP-1998-144-001,Fahmi,admin
1,EMP-9316-263-002,Ahmad,qc_packing
2,EMP-4124-636-003,Farhan,warehouse


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/dim_employees.csv"
dim_employees_df.to_csv(path_file)

## Dim Dates

In [ ]:
SIM_START = date(2021, 8, 3)
SIM_END   = date(2025, 8, 3)

# 1. Generate rentang tanggal setiap hari
date_range = pd.date_range(start=SIM_START, end=SIM_END)
dim_date = pd.DataFrame({'date': date_range})

# 2. Ekstrak komponen waktu dasar (Berguna untuk filter Tahun/Bulan di Tableau)
dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.month
dim_date['day'] = dim_date['date'].dt.day
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['day_name'] = dim_date['date'].dt.day_name()
dim_date['month_name'] = dim_date['date'].dt.month_name()
dim_date['is_weekend'] = dim_date['date'].dt.dayofweek.isin([5, 6]).astype(int) # 1 jika Sabtu/Minggu

# 3. Logic untuk Double Dates (Promo 1.1, 2.2, 3.3, ... 12.12)
# Jika nilai 'day' sama dengan 'month', maka itu Double Date
dim_date['is_double_date'] = (dim_date['day'] == dim_date['month']).astype(int)

# Opsional: Beri nama double date-nya biar bagus di dashboard (contoh: "Promo 11.11")
dim_date['event_promo'] = dim_date.apply(
    lambda row: f"Promo {row['day']}.{row['month']}" if row['is_double_date'] == 1 else "-",
    axis=1
)

# 4. Ambil Data Hari Libur Nasional Indonesia
# Mengambil hari libur dari tahun 2021 sampai 2025
id_holidays = holidays.Indonesia(years=range(SIM_START.year, SIM_END.year + 1))

# Masukkan nama hari libur ke kolom. Jika bukan hari libur, isi dengan "-"
dim_date['holiday_name'] = dim_date['date'].apply(lambda x: id_holidays.get(x, '-'))

# Buat flag/penanda (1 jika libur, 0 jika tidak)
dim_date['is_holiday'] = (dim_date['holiday_name'] != '-').astype(int)

# 5. Format kolom date menjadi string YYYY-MM-DD agar aman saat di-export ke CSV/Tableau
dim_date['date'] = dim_date['date'].dt.strftime('%Y-%m-%d')

dim_date

,date,year,month,day,quarter,day_name,month_name,is_weekend,is_double_date,event_promo,holiday_name,is_holiday
0,2021-08-03,2021,8,3,3,Tuesday,August,0,0,-,-,0
1,2021-08-04,2021,8,4,3,Wednesday,August,0,0,-,-,0
2,2021-08-05,2021,8,5,3,Thursday,August,0,0,-,-,0
3,2021-08-06,2021,8,6,3,Friday,August,0,0,-,-,0
4,2021-08-07,2021,8,7,3,Saturday,August,1,0,-,-,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1457,2025-07-30,2025,7,30,3,Wednesday,July,0,0,-,-,0
1458,2025-07-31,2025,7,31,3,Thursday,July,0,0,-,-,0
1459,2025-08-01,2025,8,1,3,Friday,August,0,0,-,-,0
1460,2025-08-02,2025,8,2,3,Saturday,August,1,0,-,-,0


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/dim_dates.csv"
dim_date.to_csv(path_file)

# Fact Star Schema

## Fact Sales

In [ ]:
# 2. Buat kolom Subtotal (Agar total revenue & pajak di Tableau nanti akurat)
temp_fact = md_fulfillment.copy()
temp_fact['subtotal_selling_price'] = temp_fact['total_quantity'] * temp_fact['product_selling_price']
temp_fact['subtotal_tax'] = temp_fact['total_quantity'] * temp_fact['product_tax']

# 3. BENTUK FACT_SALES (Hanya ambil ID dan Angka, buang nama produk/kategori/supplier)
fact_sales = temp_fact[[
    # --- Foreign Keys & Identifier ---
    'order_id',
    'product_id',        # Relasi ke dim_product
    'store_name',        # Relasi ke dim_store (karena kamu punya store_name)
    'qc_employee_id',    # Relasi ke dim_employee
    'date',              # Relasi ke dim_date

    # --- Degenerate Dimensions (Filter spesifik transaksi) ---
    'catalog_title',
    'status_order',
    'final_qc_status',

    # --- Metrics (Angka) ---
    'total_quantity',
    'product_selling_price',  # Harga Satuan
    'product_cost_price',     # Modal Satuan
    'product_tax',            # Pajak Satuan
    'subtotal_selling_price', # Revenue Asli per baris (Harga Satuan x Qty)
    'subtotal_tax',           # Pajak Asli per baris (Pajak Satuan x Qty)
    'total_qc_attempts',
    'total_rejected_qty'
]].copy()

# 4. Format tanggal agar terbaca rapi di Tableau
fact_sales['date'] = pd.to_datetime(fact_sales['date']).dt.strftime('%Y-%m-%d')

# Cek hasil akhirnya
display(fact_sales.head())

,order_id,product_id,store_name,qc_employee_id,date,catalog_title,status_order,final_qc_status,total_quantity,product_selling_price,product_cost_price,product_tax,subtotal_selling_price,subtotal_tax,total_qc_attempts,total_rejected_qty
0,ORD-1000-102-29285,PRD-8003-895-020,Bogo Helmet,EMP-9316-263-002,2022-01-11,helm bogo grey gloss + flat clear,completed,pass,1,33600.0,24000.0,4032.0,33600.0,4032.0,1,0
1,ORD-1000-102-29285,PRD-8874-102-001,Bogo Helmet,EMP-9316-263-002,2022-01-11,helm bogo grey gloss + flat clear,completed,pass,1,270200.0,193000.0,32424.0,270200.0,32424.0,1,0
2,ORD-1000-111-122718,PRD-6900-776-019,Bogo Store,EMP-9316-263-002,2023-03-29,helm bogo sage + cembung smoke,completed,pass,1,33600.0,24000.0,4032.0,33600.0,4032.0,1,0
3,ORD-1000-111-122718,PRD-7442-689-005,Bogo Store,EMP-9316-263-002,2023-03-29,helm bogo sage + cembung smoke,completed,pass,1,270200.0,193000.0,32424.0,270200.0,32424.0,1,0
4,ORD-1000-111-122718,PRD-8774-995-029,Bogo Store,EMP-9316-263-002,2023-03-29,helm hijab bogo hitam doff,completed,pass,1,282800.0,202000.0,33936.0,282800.0,33936.0,1,0


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/fact_sales.csv"
fact_sales.to_csv(path_file)

## Fact Return

In [ ]:
# 1. Filter khusus pesanan yang benar-benar diretur
temp_returns = md_after_sales[md_after_sales['is_returned'] == 1].copy()

# 2. Kalkulasi Metrik Finansial
# Refund: Uang yang kembali ke pelanggan
temp_returns['refund_amount'] = temp_returns['qty_returned'] * temp_returns['product_selling_price']

# Sunk Cost: Kerugian modal barang (Jika kondisi barang rusak/tidak bisa dijual lagi)
# Di datamu nilai restockable adalah 'yes'/'no'
temp_returns['sunk_cost_loss'] = temp_returns.apply(
    lambda row: row['qty_returned'] * row['product_cost_price'] if row['restockable'] == 'no' else 0,
    axis=1
)

# 3. BENTUK FACT_RETURNS (Format Star Schema)
fact_returns = temp_returns[[
    # --- Foreign Keys ---
    'return_id',
    'order_id',
    'product_id',
    'store_name',
    'return_date',

    # --- Degenerate Dimensions ---
    'return_reason',
    'condition',
    'restockable',
    'return_status',

    # --- Metrics ---
    'qty_returned',
    'product_selling_price',
    'product_cost_price',
    'refund_amount',
    'sunk_cost_loss'
]].copy()

# 4. Format Tanggal
fact_returns['return_date'] = pd.to_datetime(fact_returns['return_date']).dt.strftime('%Y-%m-%d')

# Verifikasi pamungkas
print("Jumlah Baris fact_returns :", len(fact_returns)) # Harus 11899
display(fact_returns.head())

Jumlah Baris fact_returns : 11899


,return_id,order_id,product_id,store_name,return_date,return_reason,condition,restockable,return_status,qty_returned,product_selling_price,product_cost_price,refund_amount,sunk_cost_loss
238,RET-9351-958-1800,ORD-1003-916-89645,PRD-8003-895-020,Bogo Store,2022-10-28,produk tidak sesuai ukuran,good,yes,received,1.0,33600.0,24000.0,33600.0,0.0
239,RET-9351-958-1800,ORD-1003-916-89645,PRD-9290-842-017,Bogo Store,2022-10-28,produk tidak sesuai ukuran,good,yes,received,1.0,270200.0,193000.0,270200.0,0.0
288,RET-5011-961-5916,ORD-1004-878-292873,PRD-7478-588-006,Bogo Store,2025-05-29,warna tidak sesuai,good,yes,received,1.0,270200.0,193000.0,270200.0,0.0
289,RET-5011-961-5916,ORD-1004-878-292873,PRD-8164-914-021,Bogo Store,2025-05-29,warna tidak sesuai,good,yes,received,1.0,33600.0,24000.0,33600.0,0.0
338,RET-1304-326-5013,ORD-1005-643-245792,PRD-8003-895-020,Bogo United,2024-10-25,warna tidak sesuai,good,yes,received,1.0,33600.0,24000.0,33600.0,0.0


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/fact_returns.csv"
fact_returns.to_csv(path_file)

## Fact Purchase

In [ ]:
# 1. Gabungkan Item PO dengan Header PO
# (Gunakan po_items_df sebagai tabel kiri agar sesuai dengan rincian barang)
temp_purchases = pd.merge(
    po_items_df,
    purchase_orders_df[['po_id', 'supplier_id', 'order_date', 'status', 'created_by']],
    how='left',
    on='po_id'
)

# 2. Buat Metrik Subtotal Kulakan
# Ini yang akan jadi "Total Pengeluaran / Modal" saat di-Sum di Tableau
temp_purchases['subtotal_purchase_cost'] = temp_purchases['quantity'] * temp_purchases['cost_price']

# 3. BENTUK FACT_PURCHASES (Star Schema)
fact_purchases = temp_purchases[[
    # --- Foreign Keys & Identifiers ---
    'po_item_id',        # Primary Key tabel ini
    'po_id',             # ID Transaksi Pembelian
    'product_id',        # Relasi ke dim_product
    'supplier_id',       # Relasi ke dim_supplier
    'created_by',        # Relasi ke dim_employee (Siapa yang order barangnya)
    'order_date',        # Relasi ke dim_date

    # --- Degenerate Dimensions ---
    'status',            # Status PO (Pending, Completed, dll)

    # --- Metrics ---
    'quantity',          # Jumlah barang yang dibeli
    'cost_price',        # Modal satuan per barang
    'subtotal_purchase_cost' # Total modal per baris
]].copy()

# 4. Format Tanggal agar Tableau bersahabat
fact_purchases['order_date'] = pd.to_datetime(fact_purchases['order_date']).dt.strftime('%Y-%m-%d')

# Cek hasilnya
print("Jumlah Baris fact_purchases :", len(fact_purchases))
display(fact_purchases.head())

Jumlah Baris fact_purchases : 5809


,po_item_id,po_id,product_id,supplier_id,created_by,order_date,status,quantity,cost_price,subtotal_purchase_cost
0,POI-8085-469-001,PO-2826-234-001,PRD-8874-102-001,SPLR-6274-501-001,EMP-1998-144-001,2021-08-03,ordered,50,193000.0,9650000.0
1,POI-1190-474-001,PO-2826-234-001,PRD-5383-617-002,SPLR-6274-501-001,EMP-1998-144-001,2021-08-03,ordered,100,193000.0,19300000.0
2,POI-9069-216-001,PO-2826-234-001,PRD-2684-647-003,SPLR-6274-501-001,EMP-1998-144-001,2021-08-03,ordered,50,193000.0,9650000.0
3,POI-9609-625-001,PO-2826-234-001,PRD-9499-652-004,SPLR-6274-501-001,EMP-1998-144-001,2021-08-03,ordered,20,193000.0,3860000.0
4,POI-6808-360-001,PO-2826-234-001,PRD-7442-689-005,SPLR-6274-501-001,EMP-1998-144-001,2021-08-03,ordered,20,193000.0,3860000.0


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/fact_purchase.csv"
fact_purchases.to_csv(path_file)

## Fact Inventory

In [ ]:
# 1. Copy data agar sumber aslinya tetap aman
temp_inv = inventory_transaction_df.copy()

# 2. Buat Logika Pengali Stok (Movement Multiplier)
# Cek isi kolom 'type' milikmu. Biasanya isinya 'IN' atau 'OUT', atau nama transaksinya.
# Silakan sesuaikan array in_types dan out_types di bawah ini dengan data riilmu:
in_types = ['in', 'masuk', 'purchase', 'return']
out_types = ['out', 'keluar', 'sales', 'order', 'reject', 'disposed']

def calculate_movement(row):
    t = str(row['type']).lower()

    # Jika tipe transaksi terdeteksi sebagai barang masuk, kalikan positif 1
    if any(keyword in t for keyword in in_types):
        return row['quantity'] * 1

    # Jika tipe transaksi terdeteksi sebagai barang keluar, kalikan negatif 1
    elif any(keyword in t for keyword in out_types):
        return row['quantity'] * -1

    # Default jika tidak terdeteksi
    else:
        return row['quantity']

# Eksekusi fungsi pengali ke dataset
temp_inv['actual_qty_movement'] = temp_inv.apply(calculate_movement, axis=1)

# 3. BENTUK FACT_INVENTORY (Star Schema)
fact_inventory = temp_inv[[
    # --- Foreign Keys ---
    'transaction_id',    # Primary Key
    'product_id',        # Relasi ke dim_product
    'actors_id',         # Relasi ke dim_employee (Siapa yang input mutasi)
    'partners_id',       # Relasi ke dim_supplier / dim_customer (Tergantung transaksinya)
    'date',              # Relasi ke dim_date

    # --- Degenerate Dimensions (Detail Dokumen & Transaksi) ---
    'type',              # Jenis mutasi ('IN' / 'OUT')
    'reference',         # Nama modul sumber (misal: 'PO', 'Sales', 'QC')
    'reference_id',      # ID Dokumen (po_id, order_id, reject_id)
    'explanation',       # Keterangan/Catatan gudang

    # --- Metrics ---
    'quantity',              # Nilai absolut (Selalu positif)
    'actual_qty_movement'    # Nilai perhitungan (+ atau - untuk kalkulasi Tableau)
]].copy()

# 4. Format Tanggal agar kompatibel dengan dim_date
fact_inventory['date'] = pd.to_datetime(fact_inventory['date']).dt.strftime('%Y-%m-%d')

# Cek hasilnya
print("Jumlah Baris fact_inventory :", len(fact_inventory))
display(fact_inventory.head())

Jumlah Baris fact_inventory : 914860


,transaction_id,product_id,actors_id,partners_id,date,type,reference,reference_id,explanation,quantity,actual_qty_movement
0,TRX-8571-882-001,PRD-8874-102-001,EMP-4124-636-003,SPLR-6274-501-001,2021-08-03,IN,purchase,PO-2826-234-001,Initial_stock,50,50
1,TRX-5224-208-002,PRD-5383-617-002,EMP-4124-636-003,SPLR-6274-501-001,2021-08-03,IN,purchase,PO-2826-234-001,Initial_stock,100,100
2,TRX-5629-903-003,PRD-2684-647-003,EMP-4124-636-003,SPLR-6274-501-001,2021-08-03,IN,purchase,PO-2826-234-001,Initial_stock,50,50
3,TRX-2322-258-004,PRD-9499-652-004,EMP-4124-636-003,SPLR-6274-501-001,2021-08-03,IN,purchase,PO-2826-234-001,Initial_stock,20,20
4,TRX-9209-447-005,PRD-7442-689-005,EMP-4124-636-003,SPLR-6274-501-001,2021-08-03,IN,purchase,PO-2826-234-001,Initial_stock,20,20


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/fact_inventory.csv"
fact_inventory.to_csv(path_file)

## Fact Rejects

In [ ]:
# 1. Filter md_fulfillment HANYA untuk barang yang mengalami reject
temp_rejects = md_fulfillment[md_fulfillment['total_rejected_qty'] > 0].copy()

# 2. Hitung Kerugian Modal (Sunk Cost Loss)
# Karena barang ini gagal dikirim/dibuang, modalnya menjadi kerugian murni
temp_rejects['sunk_cost_loss'] = temp_rejects['total_rejected_qty'] * temp_rejects['product_cost_price']

# 3. BENTUK FACT_REJECTS FINAL (Sesuai standar Star Schema)
fact_rejects = temp_rejects[[
    # --- Foreign Keys ---
    'order_id',
    'product_id',        # Relasi ke dim_product
    'store_name',        # Relasi ke dim_store
    'qc_employee_id',    # Relasi ke dim_employee (Karyawan QC yang mengecek)
    'date',              # Relasi ke dim_date (Tanggal pesanan/QC)

    # --- Degenerate Dimensions ---
    'catalog_title',     # Berguna untuk filter paket yang rawan reject
    'final_qc_status',
    'reject_reasons',    # Alasan yang sudah digabung (, ) dari kodemu
    'reject_status',     # Status yang sudah digabung (, ) dari kodemu

    # --- Metrics ---
    'total_qc_attempts', # Berapa kali bolak-balik di-packing sebelum akhirnya menyerah
    'total_rejected_qty',# Jumlah fisik yang rusak/gagal
    'product_cost_price',# Harga modal satuan
    'sunk_cost_loss'     # Total kerugian rupiah
]].copy()

# 4. Format Tanggal agar Tableau bersahabat
fact_rejects['date'] = pd.to_datetime(fact_rejects['date']).dt.strftime('%Y-%m-%d')

# Cek hasilnya
print("Jumlah Baris di fact_rejects :", len(fact_rejects))
display(fact_rejects.head())

Jumlah Baris di fact_rejects : 174958


,order_id,product_id,store_name,qc_employee_id,date,catalog_title,final_qc_status,reject_reasons,reject_status,total_qc_attempts,total_rejected_qty,product_cost_price,sunk_cost_loss
12,ORD-1000-283-197246,PRD-8983-302-018,Bogo Store,EMP-9316-263-002,2024-03-10,accessories flat smoke,pass,ada noda yg tidak bisa hilang,disposed,2,1,24000.0,24000.0
14,ORD-1000-299-250298,PRD-8665-718-015,Bogo Helmet,EMP-9316-263-002,2024-11-12,helm bogo darkgrey gloss + flat rainbow,pass,Komponen produk rusak,replacement_by_supplier,2,1,193000.0,193000.0
19,ORD-1000-355-17224,PRD-3582-781-040,Bogo Helmet,EMP-9316-263-002,2021-11-20,helm cakil hitam gloss + cembung smoke,pass,part produk tidak berfungsi,replacement_by_supplier,2,1,191000.0,191000.0
20,ORD-1000-355-17224,PRD-6689-748-014,Bogo Helmet,EMP-9316-263-002,2021-11-20,helm bogo silver + flat clear,pass,part produk tidak berfungsi,replacement_by_supplier,2,1,193000.0,193000.0
27,ORD-1000-437-80984,PRD-8983-302-018,Bogo Helmet,EMP-9316-263-002,2022-09-12,helm bogo putih gloss + flat smoke,pass,ada noda yg tidak bisa hilang,disposed,2,1,24000.0,24000.0


In [ ]:
path_file = "/content/drive/MyDrive/My Project Colab/data sales umkm/dataset merging/fact_rejects.csv"
fact_rejects.to_csv(path_file)

In [ ]:
# Mengumpulkan seluruh Dimension dan Fact Tables ke dalam satu dictionary
tables_master = {
    "dim_products": dim_products_df,
    "dim_stores": dim_store_df,
    "dim_employees": dim_employees_df,
    "dim_dates": dim_date,
    "fact_sales": fact_sales,
    "fact_returns": fact_returns,
    "fact_purchase": fact_purchases,
    "fact_inventory": fact_inventory,
    "fact_rejects": fact_rejects,
    "master_data_orders": master_data_orders,
    "master_qc": master_qc_df,
    "md_fulfillment": md_fulfillment,
    "md_after_sales": md_after_sales,
    'mdit_orders': mdit_orders,
    'mdit_returns' : mdit_return,
    'mdit_purchase' : mdit_purchase,
    'mdit_rejects' : mdit_rejects,
}

# Menampilkan list tabel yang tersedia di dalam dictionary
print("Daftar tabel dalam tables_master:")
for key in tables_master.keys():
    print(f"- {key}")

In [ ]:
fact_sales.subtotal_selling_price.sum()

np.float64(94861011000.0)